In [2]:
import pandas as pd

df = pd.read_csv("../data/games.csv")
df.columns

Index(['GAME_DATE_EST', 'GAME_ID', 'GAME_STATUS_TEXT', 'HOME_TEAM_ID',
       'VISITOR_TEAM_ID', 'SEASON', 'TEAM_ID_home', 'PTS_home', 'FG_PCT_home',
       'FT_PCT_home', 'FG3_PCT_home', 'AST_home', 'REB_home', 'TEAM_ID_away',
       'PTS_away', 'FG_PCT_away', 'FT_PCT_away', 'FG3_PCT_away', 'AST_away',
       'REB_away', 'HOME_TEAM_WINS'],
      dtype='str')

## Part 1: Basline Model

### 1. Problem Framing

In this project I build a machine learning model to predict NBA game outcomes—specifically whether the home team wins.

It is a binary classification problem:

1 → home team wins
0 → home team loses

The target variable is `HOME_TEAM_WINS`.

The model learns from historical game data to try to predict games from the test data set. This task is useful for sports analytics, betting, and performance analysis. I follow a standard ML pipeline: data loading, preprocessing, training, evaluation, and interpretation.

### 2. Load & Explore Data

The dataset contains 26,651 NBA games with 21 features, including team details, outcomes, and game statistics. The target variable, `HOME_TEAM_WINS`, indicates whether the home team won (1) or lost (0).

Most features are numerical, with some categorical variables (e.g., team IDs, game status). There are 99 missing values across several statistical columns.

Many variables (e.g., points, shooting percentages, rebounds, assists) are recorded after games, which would cause data leakage if used for prediction. To avoid this, only pre-game features—such as team identifiers and season information—are retained, while post-game statistics are excluded.

This ensures the model predicts outcomes based solely on information available before the game.

In [2]:
df.head()

,GAME_DATE_EST,GAME_ID,GAME_STATUS_TEXT,HOME_TEAM_ID,VISITOR_TEAM_ID,SEASON,TEAM_ID_home,PTS_home,FG_PCT_home,FT_PCT_home,...,AST_home,REB_home,TEAM_ID_away,PTS_away,FG_PCT_away,FT_PCT_away,FG3_PCT_away,AST_away,REB_away,HOME_TEAM_WINS
0,2022-12-22,22200477,Final,1610612740,1610612759,2022,1610612740,126.0,0.484,0.926,...,25.0,46.0,1610612759,117.0,0.478,0.815,0.321,23.0,44.0,1
1,2022-12-22,22200478,Final,1610612762,1610612764,2022,1610612762,120.0,0.488,0.952,...,16.0,40.0,1610612764,112.0,0.561,0.765,0.333,20.0,37.0,1
2,2022-12-21,22200466,Final,1610612739,1610612749,2022,1610612739,114.0,0.482,0.786,...,22.0,37.0,1610612749,106.0,0.470,0.682,0.433,20.0,46.0,1
3,2022-12-21,22200467,Final,1610612755,1610612765,2022,1610612755,113.0,0.441,0.909,...,27.0,49.0,1610612765,93.0,0.392,0.735,0.261,15.0,46.0,1
4,2022-12-21,22200468,Final,1610612737,1610612741,2022,1610612737,108.0,0.429,1.000,...,22.0,47.0,1610612741,110.0,0.500,0.773,0.292,20.0,47.0,0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 26651 entries, 0 to 26650
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   GAME_DATE_EST     26651 non-null  str    
 1   GAME_ID           26651 non-null  int64  
 2   GAME_STATUS_TEXT  26651 non-null  str    
 3   HOME_TEAM_ID      26651 non-null  int64  
 4   VISITOR_TEAM_ID   26651 non-null  int64  
 5   SEASON            26651 non-null  int64  
 6   TEAM_ID_home      26651 non-null  int64  
 7   PTS_home          26552 non-null  float64
 8   FG_PCT_home       26552 non-null  float64
 9   FT_PCT_home       26552 non-null  float64
 10  FG3_PCT_home      26552 non-null  float64
 11  AST_home          26552 non-null  float64
 12  REB_home          26552 non-null  float64
 13  TEAM_ID_away      26651 non-null  int64  
 14  PTS_away          26552 non-null  float64
 15  FG_PCT_away       26552 non-null  float64
 16  FT_PCT_away       26552 non-null  float64
 17  FG3_

In [4]:
df.describe()

,GAME_ID,HOME_TEAM_ID,VISITOR_TEAM_ID,SEASON,TEAM_ID_home,PTS_home,FG_PCT_home,FT_PCT_home,FG3_PCT_home,AST_home,REB_home,TEAM_ID_away,PTS_away,FG_PCT_away,FT_PCT_away,FG3_PCT_away,AST_away,REB_away,HOME_TEAM_WINS
count,2.665100e+04,2.665100e+04,2.665100e+04,26651.000000,2.665100e+04,26552.000000,26552.000000,26552.000000,26552.000000,26552.000000,26552.000000,2.665100e+04,26552.000000,26552.000000,26552.000000,26552.000000,26552.000000,26552.000000,26651.000000
mean,2.175487e+07,1.610613e+09,1.610613e+09,2012.113879,1.610613e+09,103.455898,0.460735,0.760377,0.356023,22.823441,43.374284,1.610613e+09,100.639876,0.449732,0.758816,0.349489,21.496271,42.113249,0.587032
std,5.570189e+06,8.638670e+00,8.659299e+00,5.587031,8.638670e+00,13.283370,0.056676,0.100677,0.111164,5.193308,6.625769,8.659299e+00,13.435868,0.055551,0.103429,0.109441,5.160596,6.533039,0.492376
min,1.030000e+07,1.610613e+09,1.610613e+09,2003.000000,1.610613e+09,36.000000,0.250000,0.143000,0.000000,6.000000,15.000000,1.610613e+09,33.000000,0.244000,0.143000,0.000000,4.000000,19.000000,0.000000
25%,2.070001e+07,1.610613e+09,1.610613e+09,2007.000000,1.610613e+09,94.000000,0.422000,0.697000,0.286000,19.000000,39.000000,1.610613e+09,91.000000,0.412000,0.692000,0.278000,18.000000,38.000000,0.000000
50%,2.120076e+07,1.610613e+09,1.610613e+09,2012.000000,1.610613e+09,103.000000,0.460000,0.765000,0.357000,23.000000,43.000000,1.610613e+09,100.000000,0.449000,0.765000,0.350000,21.000000,42.000000,1.000000
75%,2.180005e+07,1.610613e+09,1.610613e+09,2017.000000,1.610613e+09,112.000000,0.500000,0.833000,0.429000,26.000000,48.000000,1.610613e+09,110.000000,0.487000,0.833000,0.419000,25.000000,46.000000,1.000000
max,5.210021e+07,1.610613e+09,1.610613e+09,2022.000000,1.610613e+09,168.000000,0.684000,1.000000,1.000000,50.000000,72.000000,1.610613e+09,168.000000,0.687000,1.000000,1.000000,46.000000,81.000000,1.000000


In [5]:
df.isnull().sum()

GAME_DATE_EST        0
GAME_ID              0
GAME_STATUS_TEXT     0
HOME_TEAM_ID         0
VISITOR_TEAM_ID      0
SEASON               0
TEAM_ID_home         0
PTS_home            99
FG_PCT_home         99
FT_PCT_home         99
FG3_PCT_home        99
AST_home            99
REB_home            99
TEAM_ID_away         0
PTS_away            99
FG_PCT_away         99
FT_PCT_away         99
FG3_PCT_away        99
AST_away            99
REB_away            99
HOME_TEAM_WINS       0
dtype: int64

### 3. Preprocessing

Now I prepare the dataset by selecting relevant features and defining the target variable.

To avoid data leakage, I only add variables that are available before the game. Factors that are only known after the game like points scored, shooting percentages, rebounds, and assists are excluded. 

The selected input features are:
- `HOME_TEAM_ID`
- `VISITOR_TEAM_ID`
- `SEASON`

The target variable is:
- `HOME_TEAM_WINS`

I treat the identifiers as categorical variables rather than numerical values.
I won't do any cleaning since the selected features aren't affected by the missing values.

In [6]:
# Select relevant features
features = ["HOME_TEAM_ID", "VISITOR_TEAM_ID", "SEASON"]
target = "HOME_TEAM_WINS"

X = df[features]
y = df[target]

# Check shape
print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (26651, 3)
Target shape: (26651,)


### 4. Train-Test Split

Here I split the dataset into a training set and a test set. As learned in the lessons I use 80% of the data for training and 20% for testing.

Since the target variable (`HOME_TEAM_WINS`) is slightly imbalanced (approximately 59% wins and 41% losses), I use a **stratified split**. Like this the training and test sets have a similar class distribution (similar amount of wins and losses).

I do this so that I get an unbiased estimate of model performance and I can ensure that the model generalises well to new data.

In [7]:
y.value_counts(normalize=True)

HOME_TEAM_WINS
1    0.587032
0    0.412968
Name: proportion, dtype: float64

In [8]:
from sklearn.model_selection import train_test_split

# Splitting data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # keeping distribution consistent 
)

# Check shapes
print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

# Check distribution
print("\nTrain distribution:\n", y_train.value_counts(normalize=True))
print("\nTest distribution:\n", y_test.value_counts(normalize=True))

Training set: (21320, 3)
Test set: (5331, 3)

Train distribution:
 HOME_TEAM_WINS
1    0.587054
0    0.412946
Name: proportion, dtype: float64

Test distribution:
 HOME_TEAM_WINS
1    0.586944
0    0.413056
Name: proportion, dtype: float64


### 5. Benchmark Model

As defined in the project brief, I use Logistic Regression to benchmark.

Since the team identifiers are categorical variables, they are one-hot encoded before modelling. The `SEASON` feature is retained as a numeric input. To ensure that the benchmark evaluation is reliable, the model is assessed using 5-fold cross-validation on the training set.

The mean cross-validation accuracy serves as the reference point for the later model comparisons.

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import numpy as np

# Define feature types
categorical_features = ["HOME_TEAM_ID", "VISITOR_TEAM_ID"]
numeric_features = ["SEASON"]

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

# Benchmark model pipeline
benchmark_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# 5-fold cross-validation on training data
cv_scores = cross_val_score(
    benchmark_model,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("Cross-validation accuracy scores:", cv_scores)
print("Mean CV accuracy:", np.mean(cv_scores))
print("Standard deviation:", np.std(cv_scores))

Cross-validation accuracy scores: [0.60459662 0.59615385 0.59193246 0.59287054 0.58724203]
Mean CV accuracy: 0.5945590994371482
Standard deviation: 0.005772090079893115


### 6. Compare Models with Cross-Validation

Now I evaluate with two additional classification models: Decision Tree and Random Forest.

These models are chosen to compare other approaches:
- **Logistic Regression** provides a simple linear baseline.
- **Decision Tree** can capture non-linear decision boundaries.
- **Random Forest** combines multiple decision trees.

I evaluated all the models using 5-fold cross-validation on the training set. I print the mean accuracy and the standard deviation for each model in the end. 

I then choose the best model based on these cross-validation results.

In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Define the models
models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]),
    "Decision Tree": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42))
    ])
}

# Evaluate all models with 5-fold cross-validation
results = {}

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy"
    )
    results[name] = {
        "scores": scores,
        "mean_accuracy": scores.mean(),
        "std_accuracy": scores.std()
    }

# Print results
for name, result in results.items():
    print(f"{name}")
    print("Scores:", result["scores"])
    print("Mean accuracy:", result["mean_accuracy"])
    print("Standard deviation:", result["std_accuracy"])
    print("-" * 50)

Logistic Regression
Scores: [0.60459662 0.59615385 0.59193246 0.59287054 0.58724203]
Mean accuracy: 0.5945590994371482
Standard deviation: 0.005772090079893115
--------------------------------------------------
Decision Tree
Scores: [0.58020638 0.5771576  0.57762664 0.57270169 0.58208255]
Mean accuracy: 0.5779549718574108
Standard deviation: 0.0031763607792377226
--------------------------------------------------
Random Forest
Scores: [0.58677298 0.58302064 0.58935272 0.57668856 0.59287054]
Mean accuracy: 0.5857410881801126
Standard deviation: 0.005554944406722472
--------------------------------------------------


#### 6.1. Model Comparison Results

The results show that Logistic Regression performs the best mean cross-validation accuracy (mean cross-validation accuracy: ~59.5%). Therefore it outperforms both Decision Tree and Random Forest models.

Suprinsingly to me, the more complex models do not improve performance. I assume that is beacause the available features (team identifiers and season) do not provide enough information for more sophisticated models to learn meaningful additional patterns.

The Decision Tree model shows the lowest performance. I assume this is because of overfitting. The Random Forest performs slightly better but still doesn't surpass the baseline set by Logistic Regression.

Therefore, in this case, I select Logistic Regression as the best model.

This shows to me that increasing model complexity does not necessarily mean better performance. Especially when the input features are limited.

### 7. Final Evaluation on Test Set

Now that I selecting Logistic Regression, it is time to train it on the full training dataset.

I then evaluate the model on the test set.

I use the test accuracy and classification report to assess the final performance of the model.

In [12]:
from sklearn.metrics import accuracy_score, classification_report

# Training logistic regression on full training data
best_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

best_model.fit(X_train, y_train)

# Prediction on test set
y_pred = best_model.predict(X_test)

# Evaluation
test_accuracy = accuracy_score(y_test, y_pred)

print("Test Accuracy:", test_accuracy)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Test Accuracy: 0.6004501969611705

Classification Report:

              precision    recall  f1-score   support

           0       0.54      0.20      0.29      2202
           1       0.61      0.88      0.72      3129

    accuracy                           0.60      5331
   macro avg       0.58      0.54      0.51      5331
weighted avg       0.58      0.60      0.54      5331



### 8. Interpretation & Conclusion

The final model shows a test accuracy of approximately 60%. this is only slightly higher than the baseline accuracy obtained by always predicting a home team win (~59%).

As we can see in the previous section the model performs well when predicting home team wins (high recall), but significantly worse when identifying games where the home team loses. This shows me that the model is biased towards predicting the majority class (since in 58.7% of cases the home team wins).

I assume this is beacause of the limited set of input features. The model only uses team identifiers and season information. This doesnt include detailed information about factors like team strength, player performance... Therefore the model relies heavily on the trend that home teams to win more often.

This confirms what we learned in the lessons: model performance depends on the quality of the input features. Even more complex models couldnt improve performance here, showing me that the limitation lies in the data rather than the model choice.

To improve performance, I would use additional features like:
- historical win rates of teams
- recent performance trends
- head-to-head statistics
- player-level information

In conclusion, while the model provides a reasonable baseline, it demonstrates that meaningful predictions would need more informative features. 

## Part 2: Improved Model

As we saw in the Interpretation & Conclusion the first model that I trained has a lot of room for improvement. this is what I will try to tackle here.

### 1. Feature Engineering & Model Improvement

To improve the model I will add more informative features that better capture team strength and performance. 

For this I build further features:

1. I calculate team win rates separately for home and away teams. These features provide a benchmark for how successful each team is historically.
2. I calculate the average points scored by each team, capturing offensive performance.

I am aware that I am calculating overall averages (includes future games). This is a small leakage, but for the sake of learning I will nevertheless continue on this route.

In [6]:
# Create a copy of the dataframe
df_fe = df.copy()

# --- 1. Team win rate ---

# Home team win rate
home_wins = df_fe.groupby("HOME_TEAM_ID")["HOME_TEAM_WINS"].mean()

# Away team win rate:
# if HOME_TEAM_WINS = 1, away team lost -> 0
# if HOME_TEAM_WINS = 0, away team won  -> 1
away_wins = 1 - df_fe.groupby("VISITOR_TEAM_ID")["HOME_TEAM_WINS"].mean()

# Map to dataframe
df_fe["home_win_rate"] = df_fe["HOME_TEAM_ID"].map(home_wins)
df_fe["away_win_rate"] = df_fe["VISITOR_TEAM_ID"].map(away_wins)

# Difference
df_fe["win_rate_diff"] = df_fe["home_win_rate"] - df_fe["away_win_rate"]

# --- 2. Average points ---

home_points = df_fe.groupby("HOME_TEAM_ID")["PTS_home"].mean()
away_points = df_fe.groupby("VISITOR_TEAM_ID")["PTS_away"].mean()

df_fe["home_avg_points"] = df_fe["HOME_TEAM_ID"].map(home_points)
df_fe["away_avg_points"] = df_fe["VISITOR_TEAM_ID"].map(away_points)

# Difference
df_fe["points_diff"] = df_fe["home_avg_points"] - df_fe["away_avg_points"]

# Preview
df_fe[[
    "home_win_rate", "away_win_rate", "win_rate_diff",
    "home_avg_points", "away_avg_points", "points_diff"
]].head()

,home_win_rate,away_win_rate,win_rate_diff,home_avg_points,away_avg_points,points_diff
0,0.546416,0.538131,0.008285,102.275618,100.635776,1.639843
1,0.656751,0.339286,0.317465,103.385057,100.248879,3.136179
2,0.601520,0.387025,0.214495,101.893362,100.922472,0.970890
3,0.541002,0.380791,0.160211,101.801143,96.806561,4.994582
4,0.570781,0.425947,0.144834,102.835414,99.184332,3.651083


### 2. Preparing the Improved Dataset

The improved dataset combines:
- The original features (`HOME_TEAM_ID`, `VISITOR_TEAM_ID`, `SEASON`)
- The new strength features (`home_win_rate`, `away_win_rate`, `win_rate_diff`)
- The new scoring features (`home_avg_points`, `away_avg_points`, `points_diff`)

The data is then split again into training and test sets using the same 80/20 split as before.

In [7]:
# Define improved feature set
improved_features = [
    "HOME_TEAM_ID",
    "VISITOR_TEAM_ID",
    "SEASON",
    "home_win_rate",
    "away_win_rate",
    "win_rate_diff",
    "home_avg_points",
    "away_avg_points",
    "points_diff"
]

target = "HOME_TEAM_WINS"

X_improved = df_fe[improved_features]
y_improved = df_fe[target]

print("Improved feature matrix shape:", X_improved.shape)
print("Target shape:", y_improved.shape)
print(X_improved.head())

Improved feature matrix shape: (26651, 9)
Target shape: (26651,)
   HOME_TEAM_ID  VISITOR_TEAM_ID  SEASON  home_win_rate  away_win_rate  \
0    1610612740       1610612759    2022       0.546416       0.538131   
1    1610612762       1610612764    2022       0.656751       0.339286   
2    1610612739       1610612749    2022       0.601520       0.387025   
3    1610612755       1610612765    2022       0.541002       0.380791   
4    1610612737       1610612741    2022       0.570781       0.425947   

   win_rate_diff  home_avg_points  away_avg_points  points_diff  
0       0.008285       102.275618       100.635776     1.639843  
1       0.317465       103.385057       100.248879     3.136179  
2       0.214495       101.893362       100.922472     0.970890  
3       0.160211       101.801143        96.806561     4.994582  
4       0.144834       102.835414        99.184332     3.651083  


In [8]:
from sklearn.model_selection import train_test_split

X_train_imp, X_test_imp, y_train_imp, y_test_imp = train_test_split(
    X_improved,
    y_improved,
    test_size=0.2,
    random_state=42,
    stratify=y_improved
)

print("Improved training set:", X_train_imp.shape)
print("Improved test set:", X_test_imp.shape)

print("\nImproved train distribution:\n", y_train_imp.value_counts(normalize=True))
print("\nImproved test distribution:\n", y_test_imp.value_counts(normalize=True))

Improved training set: (21320, 9)
Improved test set: (5331, 9)

Improved train distribution:
 HOME_TEAM_WINS
1    0.587054
0    0.412946
Name: proportion, dtype: float64

Improved test distribution:
 HOME_TEAM_WINS
1    0.586944
0    0.413056
Name: proportion, dtype: float64


### 3. Improved Model with Logistic Regression

As before I first train on Logistic Regression but now with the improved feature set.

I use the same preprocessing and evaluation procedure:
- categorical encoding for team IDs
- adding the new numerical features
- 5-fold cross-validation on the training set

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import numpy as np

# Feature groups
categorical_features_imp = ["HOME_TEAM_ID", "VISITOR_TEAM_ID"]

numeric_features_imp = [
    "SEASON",
    "home_win_rate", "away_win_rate", "win_rate_diff",
    "home_avg_points", "away_avg_points", "points_diff"
]

# Preprocessor
preprocessor_imp = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features_imp),
        ("num", "passthrough", numeric_features_imp)
    ]
)

# Improved Logistic Regression model
model_imp = Pipeline([
    ("preprocessor", preprocessor_imp),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Cross-validation
cv_scores_imp = cross_val_score(
    model_imp,
    X_train_imp,
    y_train_imp,
    cv=5,
    scoring="accuracy"
)

print("Improved CV scores:", cv_scores_imp)
print("Improved mean accuracy:", np.mean(cv_scores_imp))
print("Improved std:", np.std(cv_scores_imp))

Improved CV scores: [0.60365854 0.59873358 0.59756098 0.59521576 0.58348968]
Improved mean accuracy: 0.5957317073170731
Improved std: 0.006713054657716629


### 4. Improved Model Comparison

Since the additional features didn't improve logistical regression now I want to see if the more complex models benefit from it.

In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Models with improved features
models_imp = {
    "Logistic Regression": model_imp,
    "Decision Tree": Pipeline([
        ("preprocessor", preprocessor_imp),
        ("classifier", DecisionTreeClassifier(random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor_imp),
        ("classifier", RandomForestClassifier(random_state=42))
    ])
}

results_imp = {}

for name, model in models_imp.items():
    scores = cross_val_score(
        model,
        X_train_imp,
        y_train_imp,
        cv=5,
        scoring="accuracy"
    )
    results_imp[name] = {
        "scores": scores,
        "mean": scores.mean(),
        "std": scores.std()
    }

# Print results
for name, res in results_imp.items():
    print(name)
    print("Scores:", res["scores"])
    print("Mean:", res["mean"])
    print("Std:", res["std"])
    print("-" * 50)

Logistic Regression
Scores: [0.60365854 0.59873358 0.59756098 0.59521576 0.58348968]
Mean: 0.5957317073170731
Std: 0.006713054657716629
--------------------------------------------------
Decision Tree
Scores: [0.56074109 0.56097561 0.58090994 0.56238274 0.57035647]
Mean: 0.5670731707317074
Std: 0.007765462332157871
--------------------------------------------------
Random Forest
Scores: [0.57551595 0.57809568 0.58419325 0.56683865 0.58677298]
Mean: 0.5782833020637899
Std: 0.00700964551087105
--------------------------------------------------


### Final Comparison and Conclusion

Here are my final findings:
Unexpectadly the new feature set does not really improve the quality of the model and Logistic Regression remains the best model with ~59.6% CV accuracy. Decision Tree and Random Forest didn't perform better than the simpler Logistical regression.

Based on this further experimentation I have learned that feature engineering must add new information to have an impact on the end results.